In [0]:
from pyspark.sql import functions as F

VOLUME = "/Volumes/workspace/gs/chegada"
CONTROLE = f"{VOLUME}/_controle"

def ingerir_bronze(fonte):
    """Auto Loader: lê só os arquivos NOVOS da subpasta, guarda como veio
    e acrescenta de onde veio cada linha e quando foi carregada."""
    (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", f"{CONTROLE}/{fonte}/schema")
        .load(f"{VOLUME}/{fonte}/")
        .withColumn("arquivo_origem", F.col("_metadata.file_path"))
        .withColumn("carregado_em", F.current_timestamp())
        .writeStream
        .option("checkpointLocation", f"{CONTROLE}/{fonte}/checkpoint")
        .trigger(availableNow=True)
        .toTable(f"workspace.gs.bronze_{fonte}")
        .awaitTermination())

In [0]:
for fonte in ["arvores", "clima", "denuncias"]:
    ingerir_bronze(fonte)
    print("ok:", fonte)

ok: arvores
ok: clima
ok: denuncias


In [0]:
%sql
SELECT 'bronze_arvores' AS tabela, COUNT(*) AS linhas, COUNT(DISTINCT arquivo_origem) AS arquivos,
       SUM(json_array_length(items)) AS registros_reais
FROM workspace.gs.bronze_arvores
UNION ALL
SELECT 'bronze_clima', COUNT(*), COUNT(DISTINCT arquivo_origem), COUNT(*)
FROM workspace.gs.bronze_clima
UNION ALL
SELECT 'bronze_denuncias', COUNT(*), COUNT(DISTINCT arquivo_origem), COUNT(*)
FROM workspace.gs.bronze_denuncias

tabela,linhas,arquivos,registros_reais
bronze_arvores,8,8,536
bronze_clima,32,8,32
bronze_denuncias,94,8,94


### Auditoria de contagens da Bronze (Requisito 05)
Rode o notebook três vezes, escolhendo a etapa no widget antes de cada execução:
1. `1o_lote` (depois do primeiro upload), 2. `2o_lote` (depois do segundo upload),
3. `reexecucao_sem_arquivos_novos` (de novo, sem arquivos novos: as contagens não podem mudar).

A tabela `bronze_auditoria` guarda as três fotografias; o `SELECT` final é o print do relatório.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS workspace.gs.bronze_auditoria (
  executado_em TIMESTAMP COMMENT 'quando a contagem foi registrada',
  etapa        STRING    COMMENT '1o_lote | 2o_lote | reexecucao_sem_arquivos_novos',
  tabela       STRING,
  linhas       BIGINT,
  arquivos     BIGINT    COMMENT 'arquivos distintos já ingeridos (arquivo_origem)'
);

In [ ]:
dbutils.widgets.dropdown("etapa", "1o_lote", ["1o_lote", "2o_lote", "reexecucao_sem_arquivos_novos"], "Etapa")
etapa = dbutils.widgets.get("etapa")

spark.sql(
    """
    INSERT INTO workspace.gs.bronze_auditoria
    SELECT current_timestamp(), :etapa, tabela, linhas, arquivos FROM (
      SELECT 'bronze_arvores' AS tabela, COUNT(*) AS linhas, COUNT(DISTINCT arquivo_origem) AS arquivos
      FROM workspace.gs.bronze_arvores
      UNION ALL
      SELECT 'bronze_clima', COUNT(*), COUNT(DISTINCT arquivo_origem) FROM workspace.gs.bronze_clima
      UNION ALL
      SELECT 'bronze_denuncias', COUNT(*), COUNT(DISTINCT arquivo_origem) FROM workspace.gs.bronze_denuncias
    )
    """,
    args={"etapa": etapa},
)
print("Contagens registradas para a etapa:", etapa)

In [ ]:
%sql
-- Print do relatório: o 2o lote aumenta as contagens; a reexecução sem arquivos novos não altera nada
SELECT etapa, tabela, linhas, arquivos, executado_em
FROM workspace.gs.bronze_auditoria
ORDER BY tabela, executado_em;